In [1]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-small")
tokenizer.pad_token = tokenizer.eos_token
tokenizer

c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


GPT2TokenizerFast(name_or_path='microsoft/DialoGPT-small', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}

In [3]:
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-small")
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
dataset = pd.read_csv("normalized_context_and_response.csv")
dataset.head()

,context,response
0,i'm going through some things with my feelings...,if everyone thinks you're worthless then maybe...
1,i'm going through some things with my feelings...,hello and thank you for your question and seek...
2,i'm going through some things with my feelings...,first thing i'd suggest is getting the sleep y...
3,i'm going through some things with my feelings...,therapy is essential for those that are feelin...
4,i'm going through some things with my feelings...,i first want to let you know that you are not ...


In [5]:
contexts = dataset['context'].astype('str').values
contexts[:5]

array(["i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthless to everyone",
       "i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthless to everyone",
       "i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthle

In [6]:
responses = dataset['response'].astype('str').values
responses[:5]

array(["if everyone thinks you're worthless then maybe you need to find new people to hang out withseriously the social context in which a person lives is a big influence in self-esteemotherwise you can go round and round trying to understand why you're not worthless then go back to the same crowd and be knocked down againthere are many inspirational messages you can find in social media \xa0maybe read some of the ones which state that no person is worthless and that everyone has a good purpose to their lifealso since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terriblebad feelings are part of living \xa0they are the motivation to remove ourselves from situations and relationships which do us more harm than goodbad feelings do feel terrible \xa0 your feeling of worthlessness may be good in the sense of motivating you to find out that you are much better than your feelings today",
       "hello and thank you for you

In [7]:
def combineText(example):
    return {
        "text": "User: " + example['context'] + 
            "Bot: " + example['response']
    }

In [8]:
def encode(example):
    return tokenizer(example['text'], truncation=True, padding=True, max_length=64)

In [9]:
def add_labels(example):
    example['labels']=example['input_ids']
    return example

In [10]:
datasets = Dataset.from_dict({
    "context": contexts,
    "response": responses
})
datasets

Dataset({
    features: ['context', 'response'],
    num_rows: 3512
})

In [11]:
datasets_split = datasets.train_test_split(test_size=0.2)
datasets_split

DatasetDict({
    train: Dataset({
        features: ['context', 'response'],
        num_rows: 2809
    })
    test: Dataset({
        features: ['context', 'response'],
        num_rows: 703
    })
})

In [12]:
trainSet = datasets_split['train']
trainSet

Dataset({
    features: ['context', 'response'],
    num_rows: 2809
})

In [13]:
testSet = datasets_split['test']
testSet

Dataset({
    features: ['context', 'response'],
    num_rows: 703
})

In [14]:
trainSet = trainSet.map(combineText)
trainSet

Map: 100%|██████████| 2809/2809 [00:00<00:00, 4012.43 examples/s]


Dataset({
    features: ['context', 'response', 'text'],
    num_rows: 2809
})

In [15]:
testSet = testSet.map(combineText)
testSet

Map: 100%|██████████| 703/703 [00:00<00:00, 3681.69 examples/s]


Dataset({
    features: ['context', 'response', 'text'],
    num_rows: 703
})

In [16]:
trainSet = trainSet.map(encode, batched=True)
trainSet

Map: 100%|██████████| 2809/2809 [00:00<00:00, 3206.07 examples/s]


Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask'],
    num_rows: 2809
})

In [17]:
testSet = testSet.map(encode, batched=True)
testSet

Map: 100%|██████████| 703/703 [00:00<00:00, 3288.78 examples/s]


Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask'],
    num_rows: 703
})

In [18]:
trainSet = trainSet.map(add_labels)
trainSet

Map: 100%|██████████| 2809/2809 [00:00<00:00, 3301.76 examples/s]


Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2809
})

In [19]:
testSet = testSet.map(add_labels)
testSet

Map: 100%|██████████| 703/703 [00:00<00:00, 3470.80 examples/s]


Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 703
})

In [20]:
trainingArgs = TrainingArguments(
    output_dir="./output",
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    evaluation_strategy='steps',
    # warmup_steps=500,
    weight_decay=0.01,
    logging_dir=None
)

In [21]:
trainSet = trainSet.select(range(1100))

In [22]:
testSet = testSet.select(range(670))

In [23]:
trainer=Trainer(
    model=model,
    args=trainingArgs,
    train_dataset= trainSet,
    eval_dataset=testSet
)

In [24]:
trainer.evaluate(testSet)

c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 11/11 [01:04<00:00,  5.87s/it]


{'eval_loss': 6.419471263885498,
 'eval_runtime': 78.1804,
 'eval_samples_per_second': 8.57,
 'eval_steps_per_second': 0.141}

In [25]:

trainer.predict(testSet.select(range(80)))

c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 2/2 [00:02<00:00,  1.10s/it]


PredictionOutput(predictions=array([[[ 25.190084,  20.328644,  19.921597, ...,  23.659145,
          22.035862,  24.333487],
        [398.70996 , 360.6712  , 359.0642  , ..., 395.01468 ,
         398.91638 , 410.08542 ],
        [362.1523  , 326.86203 , 320.04373 , ..., 358.0688  ,
         361.63928 , 370.24506 ],
        ...,
        [394.58002 , 353.23868 , 353.7964  , ..., 389.91013 ,
         394.1538  , 413.1651  ],
        [404.5524  , 360.5165  , 361.04745 , ..., 399.4514  ,
         403.95734 , 421.7322  ],
        [413.97247 , 366.0933  , 366.91034 , ..., 404.44147 ,
         410.133   , 429.0277  ]],

       [[ 25.190084,  20.328644,  19.921597, ...,  23.659145,
          22.035862,  24.333487],
        [398.70996 , 360.6712  , 359.0642  , ..., 395.01468 ,
         398.91638 , 410.08542 ],
        [362.1523  , 326.86203 , 320.04373 , ..., 358.0688  ,
         361.63928 , 370.24506 ],
        ...,
        [425.8944  , 382.91476 , 378.88217 , ..., 420.57788 ,
         425.5218

In [26]:
trainer.train()

100%|██████████| 36/36 [28:57<00:00, 48.27s/it] 

{'train_runtime': 1737.6063, 'train_samples_per_second': 1.266, 'train_steps_per_second': 0.021, 'train_loss': 5.288313971625434, 'epoch': 2.0}


TrainOutput(global_step=36, training_loss=5.288313971625434, metrics={'train_runtime': 1737.6063, 'train_samples_per_second': 1.266, 'train_steps_per_second': 0.021, 'total_flos': 71855308800000.0, 'train_loss': 5.288313971625434, 'epoch': 2.0})

In [27]:
trainer.save_model("mental-health-dialogpt")

In [28]:
trainer.evaluate(testSet)

c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 11/11 [01:23<00:00,  7.59s/it]


{'eval_loss': 4.254007339477539,
 'eval_runtime': 91.7075,
 'eval_samples_per_second': 7.306,
 'eval_steps_per_second': 0.12,
 'epoch': 2.0}

In [29]:
trainer.predict(testSet.select(range(80)))

100%|██████████| 2/2 [00:05<00:00,  2.90s/it]


PredictionOutput(predictions=array([[[ 25.24406 ,  20.362602,  19.935255, ...,  23.59983 ,
          21.976215,  24.345623],
        [309.05075 , 282.32623 , 277.66772 , ..., 309.04352 ,
         312.20865 , 322.577   ],
        [331.07526 , 302.13174 , 296.12918 , ..., 329.902   ,
         336.8476  , 341.6684  ],
        ...,
        [308.21262 , 286.18918 , 281.83606 , ..., 307.9867  ,
         310.70883 , 321.33154 ],
        [341.5949  , 311.69833 , 308.11765 , ..., 336.33142 ,
         340.10712 , 359.07965 ],
        [329.34213 , 296.5484  , 295.2171  , ..., 320.97424 ,
         327.0835  , 337.3223  ]],

       [[ 25.24406 ,  20.362602,  19.935255, ...,  23.59983 ,
          21.976215,  24.345623],
        [309.05075 , 282.32623 , 277.66772 , ..., 309.04352 ,
         312.20865 , 322.577   ],
        [331.07526 , 302.13174 , 296.12918 , ..., 329.902   ,
         336.8476  , 341.6684  ],
        ...,
        [340.55304 , 312.12097 , 306.21292 , ..., 339.97925 ,
         343.5252